# Test Pipeline RAG - Qdrant Cloud Inference

Notebook de test sur un echantillon reduit pour valider chaque etape :
1. Connexion Qdrant + chargement echantillon
2. Creation de la collection (dense + sparse)
3. Ingestion avec Cloud Inference (server-side embedding)
4. Recherche dense (semantique)
5. Recherche sparse (BM25 / lexicale)
6. Recherche hybride (RRF)
7. Recherche avec filtres metadonnees

## 1. Setup

In [5]:
import os
import ast
import pandas as pd
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

load_dotenv()

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    cloud_inference=True
)

# Verification connexion
print(client.get_collections())

collections=[CollectionDescription(name='my_first_collection'), CollectionDescription(name='tickets_test')]


In [2]:
# Chargement du dataset clean, filtre EN, echantillon de 100 tickets
df = pd.read_csv("../data/processed/tickets_clean.csv")
df["tags"] = df["tags"].apply(ast.literal_eval)

df_en = df[df["language"] == "en"].sample(n=100, random_state=42).reset_index(drop=True)

print(f"Echantillon : {len(df_en)} tickets EN")
df_en[["subject", "type", "queue", "priority", "tags"]].head()

Echantillon : 100 tickets EN


,subject,type,queue,priority,tags
0,Request for Documentation on Integrating Adobe...,Request,Product Support,low,"[Documentation, Feature, IT, Tech Support]"
1,Support Concerning Security Incident,Incident,Technical Support,high,"[Security, IT, Tech Support, Bug]"
2,Incident of Data Breach at Hospital,Incident,IT Support,high,"[Security, Bug, Performance, Virus, IT, Tech S..."
3,Assistance with Marketing Agency Integration,Problem,Product Support,low,"[Integration, Compatibility, Tech Support, Ass..."
4,Significant Decline in Digital Engagement Metr...,Incident,Product Support,medium,"[Performance, Feedback, IT, Tech Support]"


## 2. Creation de la collection

Deux vecteurs nommes :
- `dense` : `sentence-transformers/all-MiniLM-L6-v2` (384 dims, cosine) -> recherche semantique
- `bm25` : `Qdrant/bm25` (sparse) -> recherche lexicale

In [12]:
COLLECTION = "tickets_test"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "Qdrant/bm25"

# Supprimer si existe deja (mode test)
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        ),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(),
    },
)

# Index sur les champs de filtrage (requis par Qdrant pour les filtres)
for field in ["type", "queue", "priority", "tags"]:
    client.create_payload_index(
        collection_name=COLLECTION,
        field_name=field,
        field_schema=models.PayloadSchemaType.KEYWORD,
    )

print(f"Collection '{COLLECTION}' creee avec index sur type, queue, priority, tags.")

Collection 'tickets_test' creee avec index sur type, queue, priority, tags.


## 3. Ingestion avec Cloud Inference

On envoie du texte brut, Qdrant genere les embeddings cote serveur.

In [13]:
BATCH_SIZE = 50

for start in range(0, len(df_en), BATCH_SIZE):
    batch = df_en.iloc[start:start + BATCH_SIZE]
    points = []

    for i, row in batch.iterrows():
        text = row["document"]
        points.append(
            models.PointStruct(
                id=i,
                vector={
                    "dense": models.Document(
                        text=text,
                        model=DENSE_MODEL,
                    ),
                    "bm25": models.Document(
                        text=text,
                        model=SPARSE_MODEL,
                    ),
                },
                payload={
                    "subject": row["subject"],
                    "body": row["body"],
                    "answer": row["answer"],
                    "type": row["type"],
                    "queue": row["queue"],
                    "priority": row["priority"],
                    "tags": row["tags"],
                },
            )
        )

    client.upsert(collection_name=COLLECTION, points=points)
    print(f"  Batch {start // BATCH_SIZE + 1} : {len(points)} points inseres")

# Verification
info = client.get_collection(COLLECTION)
print(f"\nTotal points dans la collection : {info.points_count}")

  Batch 1 : 50 points inseres
  Batch 2 : 50 points inseres

Total points dans la collection : 100


## 4. Recherche dense (semantique)

Recherche par similarite de sens. Trouve des tickets proches conceptuellement meme si les mots-cles exacts ne sont pas presents.

In [7]:
query = "printer not connecting to laptop"

dense_results = client.query_points(
    collection_name=COLLECTION,
    query=models.Document(text=query, model=DENSE_MODEL),
    using="dense",
    limit=5,
)

print(f"Query : '{query}'\n")
for pt in dense_results.points:
    print(f"  [{pt.score:.4f}] {pt.payload['subject']}")
    print(f"           type={pt.payload['type']}  queue={pt.payload['queue']}  tags={pt.payload['tags']}")
    print()

Query : 'printer not connecting to laptop'

  [0.2384] Immediate Help Needed for Video Conference System Connectivity Issues
           type=Incident  queue=Customer Service  tags=['Network', 'Connectivity', 'Hardware', 'Tech Support']

  [0.2080] Concern Regarding Project Management
           type=Problem  queue=IT Support  tags=['Bug', 'Network', 'Performance', 'IT', 'Tech Support']

  [0.2012] Problem with HubSpot Integration
           type=Incident  queue=IT Support  tags=['Bug', 'Performance', 'IT', 'Tech Support']

  [0.1669] Detected Unauthorized Access Attempt in Hospital Systems
           type=Incident  queue=Technical Support  tags=['Security', 'Network', 'Performance', 'Bug', 'IT']

  [0.1639] Enhancing Google Meet for Team Collaboration
           type=Request  queue=Technical Support  tags=['Feature', 'Feedback', 'Documentation', 'IT', 'Tech Support']



## 5. Recherche sparse / BM25 (lexicale)

Recherche par correspondance de mots-cles. Performante pour des termes techniques exacts.

In [8]:
sparse_results = client.query_points(
    collection_name=COLLECTION,
    query=models.Document(text=query, model=SPARSE_MODEL),
    using="bm25",
    limit=5,
)

print(f"Query : '{query}'\n")
for pt in sparse_results.points:
    print(f"  [{pt.score:.4f}] {pt.payload['subject']}")
    print(f"           type={pt.payload['type']}  queue={pt.payload['queue']}  tags={pt.payload['tags']}")
    print()

Query : 'printer not connecting to laptop'

  [1.8723] Immediate Help Needed for Video Conference System Connectivity Issues
           type=Incident  queue=Customer Service  tags=['Network', 'Connectivity', 'Hardware', 'Tech Support']

  [1.7848] Data Delay Issue Encountered Today
           type=Incident  queue=Technical Support  tags=['Performance', 'Network', 'Outage', 'Disruption', 'Recovery']

  [1.7573] Multiple Product Disruptions Impacting Services
           type=Incident  queue=Technical Support  tags=['Disruption', 'Product', 'Network', 'Security', 'Outage', 'Performance', 'Tech Support']

  [1.7402] Compatibility Problem with ActiveCampaign and Sage Accounting Integration
           type=Problem  queue=Technical Support  tags=['Bug', 'Performance', 'Documentation', 'Tech Support']

  [1.6096] Irregular Investment Analysis Noted for Today
           type=Incident  queue=Product Support  tags=['Bug', 'Performance', 'Documentation', 'IT', 'Tech Support']



## 6. Recherche hybride (RRF)

Combine dense + sparse via Reciprocal Rank Fusion.  
Beneficie a la fois de la comprehension semantique et de la precision lexicale.

In [9]:
hybrid_results = client.query_points(
    collection_name=COLLECTION,
    prefetch=[
        models.Prefetch(
            query=models.Document(text=query, model=DENSE_MODEL),
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.Document(text=query, model=SPARSE_MODEL),
            using="bm25",
            limit=20,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)

print(f"Query : '{query}'\n")
for pt in hybrid_results.points:
    print(f"  [{pt.score:.4f}] {pt.payload['subject']}")
    print(f"           type={pt.payload['type']}  queue={pt.payload['queue']}  tags={pt.payload['tags']}")
    print()

Query : 'printer not connecting to laptop'

  [1.0000] Immediate Help Needed for Video Conference System Connectivity Issues
           type=Incident  queue=Customer Service  tags=['Network', 'Connectivity', 'Hardware', 'Tech Support']

  [0.4242] Concern Regarding Project Management
           type=Problem  queue=IT Support  tags=['Bug', 'Network', 'Performance', 'IT', 'Tech Support']

  [0.3929] Problem with HubSpot Integration
           type=Incident  queue=IT Support  tags=['Bug', 'Performance', 'IT', 'Tech Support']

  [0.3333] Data Delay Issue Encountered Today
           type=Incident  queue=Technical Support  tags=['Performance', 'Network', 'Outage', 'Disruption', 'Recovery']

  [0.3167] Multiple Product Disruptions Impacting Services
           type=Incident  queue=Technical Support  tags=['Disruption', 'Product', 'Network', 'Security', 'Outage', 'Performance', 'Tech Support']



## 7. Recherche avec filtres metadonnees

On peut combiner la recherche hybride avec des filtres sur les champs du payload.

In [14]:
# Recherche hybride filtree sur type=Incident et priority=high
filtered_results = client.query_points(
    collection_name=COLLECTION,
    prefetch=[
        models.Prefetch(
            query=models.Document(text=query, model=DENSE_MODEL),
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.Document(text=query, model=SPARSE_MODEL),
            using="bm25",
            limit=20,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="type",
                match=models.MatchValue(value="Incident"),
            ),
            models.FieldCondition(
                key="priority",
                match=models.MatchValue(value="high"),
            ),
        ]
    ),
    limit=5,
)

print(f"Query : '{query}' + filtre type=Incident, priority=high\n")
for pt in filtered_results.points:
    print(f"  [{pt.score:.4f}] {pt.payload['subject']}")
    print(f"           type={pt.payload['type']}  priority={pt.payload['priority']}  tags={pt.payload['tags']}")
    print()

Query : 'printer not connecting to laptop' + filtre type=Incident, priority=high

  [0.6667] Multiple Product Disruptions Impacting Services
           type=Incident  priority=high  tags=['Disruption', 'Product', 'Network', 'Security', 'Outage', 'Performance', 'Tech Support']

  [0.5000] Detected Unauthorized Access Attempt in Hospital Systems
           type=Incident  priority=high  tags=['Security', 'Network', 'Performance', 'Bug', 'IT']

  [0.4444] Multiple Tools Experiencing System Performance Issues
           type=Incident  priority=high  tags=['Performance', 'Network', 'IT', 'Tech Support']

  [0.3500] Medical Data Breach Event
           type=Incident  priority=high  tags=['Security', 'Data Breach', 'Hardware', 'Software', 'Virus']

  [0.3333] Unable to Access the Account Management Portal
           type=Incident  priority=high  tags=['Account', 'Login', 'Tech Support', 'Security']



In [15]:
# Filtre par tag
tag_results = client.query_points(
    collection_name=COLLECTION,
    prefetch=[
        models.Prefetch(
            query=models.Document(text=query, model=DENSE_MODEL),
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.Document(text=query, model=SPARSE_MODEL),
            using="bm25",
            limit=20,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="tags",
                match=models.MatchValue(value="Security"),
            ),
        ]
    ),
    limit=5,
)

print(f"Query : '{query}' + filtre tags contient 'Security'\n")
for pt in tag_results.points:
    print(f"  [{pt.score:.4f}] {pt.payload['subject']}")
    print(f"           tags={pt.payload['tags']}")
    print()

Query : 'printer not connecting to laptop' + filtre tags contient 'Security'

  [0.7000] Multiple Product Disruptions Impacting Services
           tags=['Disruption', 'Product', 'Network', 'Security', 'Outage', 'Performance', 'Tech Support']

  [0.5000] Detected Unauthorized Access Attempt in Hospital Systems
           tags=['Security', 'Network', 'Performance', 'Bug', 'IT']

  [0.4583] Medical Data Breach Event
           tags=['Security', 'Data Breach', 'Hardware', 'Software', 'Virus']

  [0.3333] Unable to Access the Account Management Portal
           tags=['Account', 'Login', 'Tech Support', 'Security']

  [0.2500] None
           tags=['Security', 'Virus', 'Disruption', 'Outage', 'Alert']



## 8. Comparaison des 3 methodes

In [16]:
def search(query_text, method="hybrid", limit=5, filters=None):
    """Fonction de recherche unifiee."""
    common = dict(
        collection_name=COLLECTION,
        limit=limit,
        query_filter=filters,
    )

    if method == "dense":
        return client.query_points(
            query=models.Document(text=query_text, model=DENSE_MODEL),
            using="dense",
            **common,
        ).points

    if method == "sparse":
        return client.query_points(
            query=models.Document(text=query_text, model=SPARSE_MODEL),
            using="bm25",
            **common,
        ).points

    # hybrid (default)
    return client.query_points(
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL),
                using="dense",
                limit=20,
            ),
            models.Prefetch(
                query=models.Document(text=query_text, model=SPARSE_MODEL),
                using="bm25",
                limit=20,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        **common,
    ).points

In [17]:
test_queries = [
    "printer not connecting to laptop",
    "billing invoice payment issue",
    "security breach unauthorized access",
]

for q in test_queries:
    print(f"{'=' * 70}")
    print(f"QUERY : {q}")
    print(f"{'=' * 70}")
    for method in ["dense", "sparse", "hybrid"]:
        results = search(q, method=method, limit=3)
        print(f"\n  [{method.upper()}]")
        for pt in results:
            print(f"    {pt.score:.4f}  {pt.payload['subject']}")
    print()

QUERY : printer not connecting to laptop

  [DENSE]
    0.2384  Immediate Help Needed for Video Conference System Connectivity Issues
    0.2080  Concern Regarding Project Management
    0.2012  Problem with HubSpot Integration

  [SPARSE]
    1.8723  Immediate Help Needed for Video Conference System Connectivity Issues
    1.7848  Data Delay Issue Encountered Today
    1.7573  Multiple Product Disruptions Impacting Services

  [HYBRID]
    1.0000  Immediate Help Needed for Video Conference System Connectivity Issues
    0.4333  Concern Regarding Project Management
    0.3929  Problem with HubSpot Integration

QUERY : billing invoice payment issue

  [DENSE]
    0.2646  None
    0.2402  Compatibility Problem with ActiveCampaign and Sage Accounting Integration
    0.2250  Unable to Access the Account Management Portal

  [SPARSE]
    1.8912  Problem with Data Loss
    1.8836  Reported Data Integration Issues
    1.8705  Compatibility Problem with ActiveCampaign and Sage Accounting Integ

In [19]:
test_queries = [
    "i have a question on compatibility problem",
    "i want to swap the product i bought last week"
]

for q in test_queries:
    print(f"{'=' * 70}")
    print(f"QUERY : {q}")
    print(f"{'=' * 70}")
    for method in ["dense", "sparse", "hybrid"]:
        results = search(q, method=method, limit=3)
        print(f"\n  [{method.upper()}]")
        for pt in results:
            print(f"    {pt.score:.4f}  {pt.payload['answer']}")
    print()

QUERY : i have a question on compatibility problem

  [DENSE]
    0.5832  Thank you for reporting the compatibility concern. To assist you further, could you please provide the specific device model names and versions of the healthcare software you are attempting to use? This information will help us verify compatibility, recommend supported configurations, and inform you about available updates or patches. If necessary, we can also schedule a call at your convenience to troubleshoot the issue. We look forward to your response.
    0.4668  Thank you for your inquiry. Our product supports integration with major CRM, email marketing, and analytics platforms through APIs and customization options. Please specify the tools you are using so we can provide you with detailed documentation and relevant case studies.
    0.4102  Examine the integration challenges by reaching out to [phone] to explore potential resolutions for compatibility issues.

  [SPARSE]
    3.4803  Dear [Name], we appreci

## Cleanup (optionnel)

In [ ]:
# Decommentez pour supprimer la collection de test
# client.delete_collection(COLLECTION)
# print(f"Collection '{COLLECTION}' supprimee.")